# 面试问题：Agent 怎样解析系统、开发者、用户和外部工具内容的指令冲突？

**一句话回答。** 把可执行指令表示为带 authority、topic、动作和来源的结构化记录；同 topic 由更高 authority 决定，同级冲突要求澄清，外部文档/工具文本默认只是数据而非指令。每个决策输出获胜与被拒指令的 provenance，副作用仍需独立权限门禁。

本 Notebook 用 Python 标准库手写最小数据合同、状态机、验证器和失败分支。断言针对受控小数据，不等于模型语义正确、数据库安全、图谱质量或生产 Agent 的安全保证。

**资料入口。** [OpenAI Model Spec 的 Chain of Command](https://model-spec.openai.com/2025-02-12.html) 说明冲突指令需要按权威层级解析；本例是通用确定性 policy 层，不模拟任何模型的自然语言理解。


In [ ]:
question = "Agent 指令层级冲突"  # 执行本行的状态、计算或校验逻辑。
assert "Agent" in question  # 执行本行的状态、计算或校验逻辑。
assert 8 // 2 == 4  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 区分可执行指令和不可信内容

系统/开发者/用户消息与网页、RAG chunk、工具返回不是同一 authority。外部文本可能包含“忽略前文”字样，但默认仍是待分析数据；把它直接拼进执行 prompt 会造成 prompt injection。


In [ ]:
authority = {"system": 3, "developer": 2, "user": 1, "external": 0}  # 执行本行的状态、计算或校验逻辑。
messages = [{"id": "s1", "source": "system", "topic": "payments", "value": "deny"}, {"id": "d1", "source": "developer", "topic": "data", "value": "read_only"}, {"id": "u1", "source": "user", "topic": "payments", "value": "allow"}, {"id": "x1", "source": "external", "topic": "payments", "value": "allow"}]  # 执行本行的状态、计算或校验逻辑。
assert authority["system"] > authority["user"]  # 执行本行的状态、计算或校验逻辑。
assert messages[-1]["source"] == "external"  # 执行本行的状态、计算或校验逻辑。
assert len(messages) == 4  # 执行本行的状态、计算或校验逻辑。

## 2. 先规范化 topic 和动作，不能靠字符串位置猜优先级

真实语言理解可由模型做候选抽取，但 policy 层应使用受控 topic/action 枚举和签名。未知 topic、模糊否定或参数缺失应进入澄清/拒绝队列，而不是默认放行。


In [ ]:
allowed_topics = {"payments", "data"}  # 执行本行的状态、计算或校验逻辑。
def valid_message(message):  # 执行本行的状态、计算或校验逻辑。
    return message["source"] in authority and message["topic"] in allowed_topics and message["value"] in {"allow", "deny", "read_only"}  # 执行本行的状态、计算或校验逻辑。
assert all(valid_message(message) for message in messages)  # 执行本行的状态、计算或校验逻辑。
assert not valid_message({"source": "user", "topic": "unknown", "value": "allow"})  # 执行本行的状态、计算或校验逻辑。
assert not valid_message({"source": "user", "topic": "data", "value": "maybe"})  # 执行本行的状态、计算或校验逻辑。

## 3. 同 topic 只比较同一语义域中的 authority

解析器按 topic 选最高 authority；若多个同级最高指令取值相反，应返回 clarify，不能用“最后一条消息”或模型置信度打破平局。external 即使语气强烈也永远不能升级 authority。


In [ ]:
def resolve(topic, records):  # 执行本行的状态、计算或校验逻辑。
    candidates = [record for record in records if record["topic"] == topic]  # 执行本行的状态、计算或校验逻辑。
    highest = max(authority[record["source"]] for record in candidates)  # 执行本行的状态、计算或校验逻辑。
    winners = [record for record in candidates if authority[record["source"]] == highest]  # 执行本行的状态、计算或校验逻辑。
    values = {record["value"] for record in winners}  # 执行本行的状态、计算或校验逻辑。
    return {"status": "apply" if len(values) == 1 else "clarify", "value": next(iter(values)) if len(values) == 1 else None, "winner_ids": tuple(record["id"] for record in winners), "discarded": tuple(record["id"] for record in candidates if record not in winners)}  # 执行本行的状态、计算或校验逻辑。
payment_policy = resolve("payments", messages)  # 执行本行的状态、计算或校验逻辑。
assert payment_policy["value"] == "deny"  # 执行本行的状态、计算或校验逻辑。
assert payment_policy["winner_ids"] == ("s1",)  # 执行本行的状态、计算或校验逻辑。
assert set(payment_policy["discarded"]) == {"u1", "x1"}  # 执行本行的状态、计算或校验逻辑。

## 4. 外部内容可作为证据，不能作为 policy 更新

Agent 可从网页/工具读到金额、订单号或说明，但外部内容不能赋予自己“允许转账”的权限。需要保留其来源/污点，让模型可引用事实，同时在 action sink 前由独立策略决定是否可用。


In [ ]:
def executable(record):  # 执行本行的状态、计算或校验逻辑。
    return record["source"] != "external" and valid_message(record)  # 执行本行的状态、计算或校验逻辑。
assert not executable(messages[-1])  # 执行本行的状态、计算或校验逻辑。
assert executable(messages[0])  # 执行本行的状态、计算或校验逻辑。
assert messages[-1]["value"] == "allow"  # 执行本行的状态、计算或校验逻辑。

## 5. 同级冲突必须澄清或使用显式业务规则

两个用户消息或两个开发者配置同时给出不同动作时，静默选择任一项都不可审计。本例返回 clarify；若业务要求“最新配置胜出”，也必须把排序依据、版本和批准人写入 policy。


In [ ]:
same_level = [{"id": "u2", "source": "user", "topic": "data", "value": "allow"}, {"id": "u3", "source": "user", "topic": "data", "value": "deny"}]  # 执行本行的状态、计算或校验逻辑。
data_policy = resolve("data", same_level)  # 执行本行的状态、计算或校验逻辑。
assert data_policy["status"] == "clarify"  # 执行本行的状态、计算或校验逻辑。
assert data_policy["value"] is None  # 执行本行的状态、计算或校验逻辑。
assert set(data_policy["winner_ids"]) == {"u2", "u3"}  # 执行本行的状态、计算或校验逻辑。

## 6. 指令解析不替代副作用授权

即使 user 在其 authority 内请求某个动作，支付、删除、发送邮件等 sink 仍需 capability、精确参数、审批票据和有效期。层级解决“听谁的”，授权解决“能否做”。


In [ ]:
action = {"topic": "payments", "amount": 100, "approval": None}  # 执行本行的状态、计算或校验逻辑。
def action_gate(policy, action_value):  # 执行本行的状态、计算或校验逻辑。
    return policy["status"] == "apply" and policy["value"] == "allow" and action_value["approval"] is not None  # 执行本行的状态、计算或校验逻辑。
assert not action_gate(payment_policy, action)  # 执行本行的状态、计算或校验逻辑。
assert not action_gate({"status": "apply", "value": "allow"}, action)  # 执行本行的状态、计算或校验逻辑。
assert action["approval"] is None  # 执行本行的状态、计算或校验逻辑。

## 7. 决策 trace 保存胜者、舍弃者与 policy 版本

当用户问“为什么没执行”时，应能给出高层约束 id 和被忽略的外部/低优先级指令 id，而不必泄露隐藏 prompt 原文。trace 还应包含 policy version 和参数 digest，支持回归与审计。


In [ ]:
decision_trace = {"topic": "payments", "status": payment_policy["status"], "winner_ids": payment_policy["winner_ids"], "discarded": payment_policy["discarded"], "policy_version": "p1"}  # 执行本行的状态、计算或校验逻辑。
assert decision_trace["winner_ids"] == ("s1",)  # 执行本行的状态、计算或校验逻辑。
assert "u1" in decision_trace["discarded"]  # 执行本行的状态、计算或校验逻辑。
assert decision_trace["policy_version"] == "p1"  # 执行本行的状态、计算或校验逻辑。

## 8. 评测同时测服从与注入抵抗

回归集应覆盖高低优先级冲突、同级冲突、引用型外部文本、越权副作用和未知 topic。指标包括正确优先级、clarify 率、external escalation 漏检、误拒率和人审负担，不能只测单轮 jailbreak 成败。


In [ ]:
tests = {"system_beats_user": payment_policy["value"] == "deny", "same_level_clarifies": data_policy["status"] == "clarify", "external_not_executable": not executable(messages[-1])}  # 执行本行的状态、计算或校验逻辑。
assert all(tests.values())  # 执行本行的状态、计算或校验逻辑。
assert len(tests) == 3  # 执行本行的状态、计算或校验逻辑。
assert resolve("data", messages)["value"] == "read_only"  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试中把指令层级、外部内容信任和副作用授权拆开回答：先规范化候选指令，按 authority 和同级冲突策略得到可审计决策；外部工具文本只当 evidence；最后每个危险 action 再经过 capability/approval gate。模型提示本身不能替代确定性权限控制。
